# NLP Lab Task 5

## Submitted By
**Musa Sajjad — SP24-BDS-039**




In [1]:
# ============================================================
# DATA
# ============================================================

train = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
    "the dog ate the bone"
]

test = [
    "the cat sat on the rug",
    "the fish sat on the mat"
]

print("Training sentences:")
for s in train:
    print(" ", s)

print("\nTest sentences:")
for s in test:
    print(" ", s)


Training sentences:
  the cat sat on the mat
  the cat ate the fish
  the dog sat on the rug
  the dog ate the bone

Test sentences:
  the cat sat on the rug
  the fish sat on the mat


## Task 1 — Tokenization

Requirements:
- lowercase everything
- split on spaces
- add `<s>` and `</s>`
- vocabulary contains all words plus `</s>` but not `<s>`


In [2]:
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]


train_tokens = [tokenize(s) for s in train]
test_tokens = [tokenize(s) for s in test]

print("Tokenized training sentences:")
for tokens in train_tokens:
    print(tokens)

print("\nTokenized test sentences:")
for tokens in test_tokens:
    print(tokens)

train_token_count_with_markers = sum(len(tokens) for tokens in train_tokens)
train_token_count_without_markers = sum(
    sum(1 for token in tokens if token not in ("<s>", "</s>"))
    for tokens in train_tokens
)

print("\nTotal training tokens with markers:", train_token_count_with_markers)
print("Total training tokens without markers:", train_token_count_without_markers)


def build_vocab(sentences):
    vocab = set()
    for sentence in sentences:
        for token in tokenize(sentence):
            if token != "<s>":
                vocab.add(token)
    return vocab


Vocab = build_vocab(train)
V = len(Vocab)

print("\nV =", V)
print("Sorted vocabulary:")
print(sorted(Vocab))


Tokenized training sentences:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']

Tokenized test sentences:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']

Total training tokens with markers: 30
Total training tokens without markers: 22

V = 11
Sorted vocabulary:
['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


## Task 2 — Generating N-grams

In [3]:
def get_ngrams(tokens, n):
    if n <= 0:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


first_sentence = train_tokens[0]

print("First training sentence:")
print(first_sentence)

print("\nUnigrams:")
print(get_ngrams(first_sentence, 1))

print("\nBigrams:")
print(get_ngrams(first_sentence, 2))

print("\nTrigrams:")
print(get_ngrams(first_sentence, 3))


print("\nVerification of m - n + 1 formula:")
for sentence_number, tokens in enumerate(train_tokens, start=1):
    m = len(tokens)
    for n in (1, 2, 3):
        expected = max(0, m - n + 1)
        actual = len(get_ngrams(tokens, n))
        print(
            f"Sentence {sentence_number}, n={n}: "
            f"m={m}, expected={expected}, actual={actual}"
        )
        assert actual == expected

print("\nAll formula checks passed.")

print("\nWhen n is larger than sentence length:")
large_n = len(first_sentence) + 5
print("n =", large_n)
print(get_ngrams(first_sentence, large_n))


First training sentence:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']

Unigrams:
[('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]

Bigrams:
[('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]

Trigrams:
[('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]

Verification of m - n + 1 formula:
Sentence 1, n=1: m=8, expected=8, actual=8
Sentence 1, n=2: m=8, expected=7, actual=7
Sentence 1, n=3: m=8, expected=6, actual=6
Sentence 2, n=1: m=7, expected=7, actual=7
Sentence 2, n=2: m=7, expected=6, actual=6
Sentence 2, n=3: m=7, expected=5, actual=5
Sentence 3, n=1: m=8, expected=8, actual=8
Sentence 3, n=2: m=8, expected=7, actual=7
Sentence 3, n=3: m=8, expected=6, actual=6
Sentence 4, n=1: m=7, expected=7, actual=7
Sentence 4, n=2: m=7, expected=6, actual=6
Sentence 4, n=3: m=7, expected=5, actual=5

All

## Task 3 — Counting N-grams

In [4]:
from collections import Counter


unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for tokens in train_tokens:
    unigram_counts.update(get_ngrams(tokens, 1))
    bigram_counts.update(get_ngrams(tokens, 2))
    trigram_counts.update(get_ngrams(tokens, 3))


def print_count_table(counter, title):
    print(f"\n{title}")
    print("-" * len(title))
    for ngram, count in counter.most_common():
        print(f"{ngram}: {count}")


print_count_table(unigram_counts, "Unigram Counts")
print_count_table(bigram_counts, "Bigram Counts")
print_count_table(trigram_counts, "Trigram Counts")

print("\n5 most common bigrams:")
for bigram, count in bigram_counts.most_common(5):
    print(bigram, "->", count)

print("\nNumber of distinct unigrams:", len(unigram_counts))
print("Number of distinct bigrams:", len(bigram_counts))
print("Number of distinct trigrams:", len(trigram_counts))


def count_of(ngram):
    if len(ngram) == 1:
        return unigram_counts[tuple(ngram)]
    elif len(ngram) == 2:
        return bigram_counts[tuple(ngram)]
    elif len(ngram) == 3:
        return trigram_counts[tuple(ngram)]
    return 0


seen_bigram = ("the", "cat")
unseen_bigram = ("fish", "dog")

print("\nSeen bigram:", seen_bigram, "Count =", count_of(seen_bigram))
print("Unseen bigram:", unseen_bigram, "Count =", count_of(unseen_bigram))

assert count_of(seen_bigram) > 0
assert count_of(unseen_bigram) == 0



Unigram Counts
--------------
('the',): 8
('<s>',): 4
('</s>',): 4
('cat',): 2
('sat',): 2
('on',): 2
('ate',): 2
('dog',): 2
('mat',): 1
('fish',): 1
('rug',): 1
('bone',): 1

Bigram Counts
-------------
('<s>', 'the'): 4
('the', 'cat'): 2
('sat', 'on'): 2
('on', 'the'): 2
('ate', 'the'): 2
('the', 'dog'): 2
('cat', 'sat'): 1
('the', 'mat'): 1
('mat', '</s>'): 1
('cat', 'ate'): 1
('the', 'fish'): 1
('fish', '</s>'): 1
('dog', 'sat'): 1
('the', 'rug'): 1
('rug', '</s>'): 1
('dog', 'ate'): 1
('the', 'bone'): 1
('bone', '</s>'): 1

Trigram Counts
--------------
('<s>', 'the', 'cat'): 2
('sat', 'on', 'the'): 2
('<s>', 'the', 'dog'): 2
('the', 'cat', 'sat'): 1
('cat', 'sat', 'on'): 1
('on', 'the', 'mat'): 1
('the', 'mat', '</s>'): 1
('the', 'cat', 'ate'): 1
('cat', 'ate', 'the'): 1
('ate', 'the', 'fish'): 1
('the', 'fish', '</s>'): 1
('the', 'dog', 'sat'): 1
('dog', 'sat', 'on'): 1
('on', 'the', 'rug'): 1
('the', 'rug', '</s>'): 1
('the', 'dog', 'ate'): 1
('dog', 'ate', 'the'): 1
('ate', 

## Task 4 — Unigram Model (MLE)

Formula:

\[
P(w) = \frac{C(w)}{N}
\]

where `N` is the total number of training tokens including markers.


In [5]:
import math
import itertools


N = sum(unigram_counts.values())


def unigram_prob(w):
    return unigram_counts[(w,)] / N


print("Unigram probabilities:")
print("-" * 35)

unigram_probability_sum = 0.0

for word in sorted(Vocab | {"<s>"}):
    p = unigram_prob(word)
    unigram_probability_sum += p
    print(f"{word:10s} {p:.4f}")

print("\nSum of unigram probabilities:", round(unigram_probability_sum, 4))
assert math.isclose(unigram_probability_sum, 1.0, rel_tol=1e-12)


def unigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    probability = 1.0
    for token in tokens:
        probability *= unigram_prob(token)
    return probability


print("\nUnigram sentence probabilities:")
for sentence in test:
    print(sentence, "->", round(unigram_sentence_prob(sentence), 4))


# Demonstrate that word order does not affect a unigram probability.
words = test[0].lower().split()
permutation_results = {}

for perm in itertools.permutations(words):
    perm_sentence = " ".join(perm)
    permutation_results[perm_sentence] = unigram_sentence_prob(perm_sentence)

unique_probabilities = set(permutation_results.values())

print("\nNumber of permutations checked:", len(permutation_results))
print("Number of unique unigram probabilities:", len(unique_probabilities))

for sentence_variant, probability in list(permutation_results.items())[:5]:
    print(sentence_variant, "->", round(probability, 8))

assert len(unique_probabilities) == 1
print("\nAll permutations have the same unigram probability.")


Unigram probabilities:
-----------------------------------
</s>       0.1333
<s>        0.1333
ate        0.0667
bone       0.0333
cat        0.0667
dog        0.0667
fish       0.0333
mat        0.0333
on         0.0667
rug        0.0333
sat        0.0667
the        0.2667

Sum of unigram probabilities: 1.0

Unigram sentence probabilities:
the cat sat on the rug -> 0.0
the fish sat on the mat -> 0.0

Number of permutations checked: 360
Number of unique unigram probabilities: 1
the cat sat on the rug -> 1e-08
the cat sat on rug the -> 1e-08
the cat sat the on rug -> 1e-08
the cat sat the rug on -> 1e-08
the cat sat rug on the -> 1e-08

All permutations have the same unigram probability.


## Task 5 — Bigram Model (MLE)

Formula:

\[
P(w_i|w_{i-1}) = \frac{C(w_{i-1},w_i)}{C(w_{i-1})}
\]


In [6]:
def bigram_prob(prev, w):
    denominator = unigram_counts[(prev,)]
    if denominator == 0:
        return 0.0
    return bigram_counts[(prev, w)] / denominator


def next_word_distribution(prev):
    denominator = unigram_counts[(prev,)]
    if denominator == 0:
        return {}

    distribution = {}
    for (p, word), count in bigram_counts.items():
        if p == prev:
            distribution[word] = count / denominator
    return distribution


the_distribution = next_word_distribution("the")

print("Next-word distribution for 'the':")
for word, probability in sorted(the_distribution.items()):
    print(f"{word:10s} {probability:.4f}")

assert math.isclose(sum(the_distribution.values()), 1.0, rel_tol=1e-12)


def bigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    probability = 1.0

    for prev, word in get_ngrams(tokens, 2):
        probability *= bigram_prob(prev, word)

    return probability


print("\nBigram sentence probabilities:")
for sentence in test:
    print(sentence, "->", round(bigram_sentence_prob(sentence), 4))


def predict_next(prev):
    distribution = next_word_distribution(prev)

    if not distribution:
        return None

    max_probability = max(distribution.values())
    candidates = [
        word for word, probability in distribution.items()
        if probability == max_probability
    ]

    return sorted(candidates)[0]


print("\nNext-word predictions:")

words_to_test = ["the", "cat", "dog", "sat", "ate", "on"]

for word in words_to_test:
    print(f"After '{word}' -> {predict_next(word)}")


Next-word distribution for 'the':
bone       0.1250
cat        0.2500
dog        0.2500
fish       0.1250
mat        0.1250
rug        0.1250

Bigram sentence probabilities:
the cat sat on the rug -> 0.0156
the fish sat on the mat -> 0.0

Next-word predictions:
After 'the' -> cat
After 'cat' -> ate
After 'dog' -> ate
After 'sat' -> on
After 'ate' -> the
After 'on' -> the


## Task 6 — Trigram Model

In [7]:
def trigram_prob(w1, w2, w3):
    denominator = bigram_counts[(w1, w2)]
    if denominator == 0:
        return 0.0
    return trigram_counts[(w1, w2, w3)] / denominator


def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)

    if len(tokens) < 2:
        return 0.0

    # First word after <s> is handled using the bigram model.
    probability = bigram_prob(tokens[0], tokens[1])

    # Remaining words are handled using the trigram model.
    for w1, w2, w3 in get_ngrams(tokens[1:], 3):
        probability *= trigram_prob(w1, w2, w3)

    return probability


print("Trigram sentence probabilities:")
for sentence in test:
    print(sentence, "->", round(trigram_sentence_prob(sentence), 4))


print("\nDistinct N-gram counts")
print("----------------------")
print(f"Unigrams : {len(unigram_counts)}")
print(f"Bigrams  : {len(bigram_counts)}")
print(f"Trigrams : {len(trigram_counts)}")


for sentence in test:
    tokens = tokenize(sentence)
    sentence_bigrams = get_ngrams(tokens, 2)
    sentence_trigrams = get_ngrams(tokens, 3)

    unseen_bigrams_list = [
        bg for bg in sentence_bigrams if bigram_counts[bg] == 0
    ]

    unseen_trigrams_list = [
        tg for tg in sentence_trigrams if trigram_counts[tg] == 0
    ]

    print("\nSentence:", sentence)
    print("Unseen bigrams:", unseen_bigrams_list)
    print("Unseen trigrams:", unseen_trigrams_list)

    if len(unseen_bigrams_list) > len(unseen_trigrams_list):
        print("Model comparison: Bigram model has more unseen N-grams.")
    elif len(unseen_trigrams_list) > len(unseen_bigrams_list):
        print("Model comparison: Trigram model has more unseen N-grams.")
    else:
        print("Model comparison: Both have the same number of unseen N-grams.")


Trigram sentence probabilities:
the cat sat on the rug -> 0.25
the fish sat on the mat -> 0.0

Distinct N-gram counts
----------------------
Unigrams : 12
Bigrams  : 18
Trigrams : 19

Sentence: the cat sat on the rug
Unseen bigrams: []
Unseen trigrams: []
Model comparison: Both have the same number of unseen N-grams.

Sentence: the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
Model comparison: Trigram model has more unseen N-grams.


## Task 7 — Zero-Probability Problem

In [8]:
def unseen_bigrams(sentence):
    tokens = tokenize(sentence)
    return [
        bigram
        for bigram in get_ngrams(tokens, 2)
        if bigram_counts[bigram] == 0
    ]


print("Unseen bigrams and their MLE probabilities:")

for sentence in test:
    unseen = unseen_bigrams(sentence)

    print("\nSentence:", sentence)

    if not unseen:
        print("No unseen bigrams.")
    else:
        for bigram in unseen:
            print(bigram, "->", f"{bigram_prob(*bigram):.4f}")

print("\nAt least one unseen bigram has probability 0 under MLE.")


# Count possible word-only pairs, excluding markers.
words_only = sorted(
    token for token in Vocab
    if token not in ("<s>", "</s>")
)

possible_word_pairs = len(words_only) ** 2
observed_word_pairs = sum(
    1
    for (w1, w2), count in bigram_counts.items()
    if w1 in words_only and w2 in words_only and count > 0
)

unseen_word_pairs = possible_word_pairs - observed_word_pairs
unseen_percentage = (
    unseen_word_pairs / possible_word_pairs * 100
    if possible_word_pairs else 0
)

print("\nPossible word-only pairs:", possible_word_pairs)
print("Observed word-only pairs:", observed_word_pairs)
print("Never observed word-only pairs:", unseen_word_pairs)
print("Percentage never observed:", f"{unseen_percentage:.2f}%")


def safe_log(p):
    return -math.inf if p == 0 else math.log(p)


second_test = test[1]
second_tokens = tokenize(second_test)

second_bigram_probabilities = [
    bigram_prob(prev, word)
    for prev, word in get_ngrams(second_tokens, 2)
]

log_probability = sum(
    safe_log(p) for p in second_bigram_probabilities
)

print("\nLog probability of second test sentence:")
print(second_test, "->", log_probability)


Unseen bigrams and their MLE probabilities:

Sentence: the cat sat on the rug
No unseen bigrams.

Sentence: the fish sat on the mat
('fish', 'sat') -> 0.0000

At least one unseen bigram has probability 0 under MLE.

Possible word-only pairs: 100
Observed word-only pairs: 13
Never observed word-only pairs: 87
Percentage never observed: 87.00%

Log probability of second test sentence:
the fish sat on the mat -> -inf


## Task 8 — Smoothing

Add-one smoothing:

\[
P(w|v) = \frac{C(v,w)+1}{C(v)+V}
\]

The same vocabulary `V` is used throughout.


In [12]:
def bigram_prob_addone(prev, w):
    return (
        bigram_counts[(prev, w)] + 1
    ) / (
        unigram_counts[(prev,)] + V
    )


comparison_pairs = [
    ("fish", "sat"),
    ("the", "cat")
]

print("MLE vs Add-One Smoothing")
print("-" * 55)
print(f"{'Probability':<20}{'MLE':>12}{'Add-One':>15}")

for prev, word in comparison_pairs:
    mle = bigram_prob(prev, word)
    addone = bigram_prob_addone(prev, word)

    print(
        f"P({word} | {prev}){'':<8}"
        f"{mle:>12.4f}{addone:>15.4f}"
    )


# Verify smoothed distributions sum to 1 for every possible previous context word.
# '</s>' is never a previous word, and '<s>' can be a previous word.
print("\nChecking normalization for all possible context words:")

history_words = (Vocab | {"<s>"}) - {"</s>"}

for prev in sorted(history_words):
    total = sum(
        bigram_prob_addone(prev, word)
        for word in Vocab
    )
    assert math.isclose(total, 1.0, rel_tol=1e-12), (
        prev, total
    )

print("All valid add-one distributions sum to 1.")


def bigram_prob_addk(prev, w, k):
    return (
        bigram_counts[(prev, w)] + k
    ) / (
        unigram_counts[(prev,)] + k * V
    )


print("\nP(sat | fish) for different k values:")
print("-" * 35)
print(f"{'k':<10}{'P(sat | fish)':>20}")

for k in (1, 0.5, 0.1, 0.01):
    probability = bigram_prob_addk("fish", "sat", k)
    print(f"{k:<10}{probability:>20.4f}")

MLE vs Add-One Smoothing
-------------------------------------------------------
Probability                  MLE        Add-One
P(sat | fish)              0.0000         0.0833
P(cat | the)              0.2500         0.1579

Checking normalization for all possible context words:
All valid add-one distributions sum to 1.

P(sat | fish) for different k values:
-----------------------------------
k                P(sat | fish)
1                       0.0833
0.5                     0.0769
0.1                     0.0476
0.01                    0.0090


## Task 9 — Perplexity and Model Comparison

Perplexity is calculated from the average negative log probability:

\[
PP = \exp\left(-\frac{1}{N}\sum_i \ln P(w_i|context_i)\right)
\]

If any probability is zero, perplexity is infinity.


In [13]:
def perplexity(sentences, prob_fn):
    log_probability_sum = 0.0
    token_count = 0

    for sentence in sentences:
        probabilities = prob_fn(sentence)

        for probability in probabilities:
            if probability == 0:
                return math.inf

            log_probability_sum += math.log(probability)
            token_count += 1

    if token_count == 0:
        return math.inf

    return math.exp(-log_probability_sum / token_count)


def bigram_probabilities_for_sentence(sentence):
    tokens = tokenize(sentence)
    return [
        bigram_prob(prev, word)
        for prev, word in get_ngrams(tokens, 2)
    ]


def bigram_addone_probabilities_for_sentence(sentence):
    tokens = tokenize(sentence)
    return [
        bigram_prob_addone(prev, word)
        for prev, word in get_ngrams(tokens, 2)
    ]


def unigram_addone_probabilities_for_sentence(sentence):
    tokens = tokenize(sentence)
    return [
        (unigram_counts[(word,)] + 1) / (N + V)
        for word in tokens
    ]


print("Perplexity comparison")
print("-" * 55)

results = [
    (
        "Bigram MLE",
        perplexity(test, bigram_probabilities_for_sentence)
    ),
    (
        "Bigram Add-One",
        perplexity(test, bigram_addone_probabilities_for_sentence)
    ),
    (
        "Unigram Add-One",
        perplexity(test, unigram_addone_probabilities_for_sentence)
    )
]

for model, value in results:
    if math.isinf(value):
        display_value = "infinity"
    else:
        display_value = f"{value:.4f}"

    print(f"{model:<22} {display_value}")


print("\nAdd-k Bigram Perplexity")
print("-" * 40)


def make_addk_probability_function(k):
    def probability_function(sentence):
        tokens = tokenize(sentence)
        return [
            bigram_prob_addk(prev, word, k)
            for prev, word in get_ngrams(tokens, 2)
        ]
    return probability_function


addk_results = []

for k in (1, 0.5, 0.1, 0.01):
    pp = perplexity(test, make_addk_probability_function(k))
    addk_results.append((k, pp))
    print(f"k = {k:<5} Perplexity = {pp:.4f}")

best_k_direct, best_pp_direct = min(
    addk_results,
    key=lambda item: item[1]
)

print(
    f"\nBest k directly on test set: "
    f"k = {best_k_direct}, perplexity = {best_pp_direct:.4f}"
)


Perplexity comparison
-------------------------------------------------------
Bigram MLE             infinity
Bigram Add-One         5.7969
Unigram Add-One        9.8614

Add-k Bigram Perplexity
----------------------------------------
k = 1     Perplexity = 5.7969
k = 0.5   Perplexity = 4.5703
k = 0.1   Perplexity = 2.9559
k = 0.01  Perplexity = 2.6272

Best k directly on test set: k = 0.01, perplexity = 2.6272


## Task 9.4 — Validation-Based Selection of k

For this experiment:
- train only on the first three training sentences
- build `V` from all four training sentences
- use the fourth training sentence as validation data
- select the best `k` using validation perplexity
- evaluate that selected `k` on the two original test sentences
- compare with the `k` selected directly on the test set


In [14]:
# ------------------------------------------------------------
# Validation experiment
# ------------------------------------------------------------

validation_train_sentences = train[:3]
validation_sentence = train[3]

# Vocabulary is deliberately built from ALL FOUR training sentences.
validation_vocab = build_vocab(train)
validation_V = len(validation_vocab)

validation_train_tokens = [
    tokenize(sentence)
    for sentence in validation_train_sentences
]

validation_unigram_counts = Counter()
validation_bigram_counts = Counter()

for tokens in validation_train_tokens:
    validation_unigram_counts.update(get_ngrams(tokens, 1))
    validation_bigram_counts.update(get_ngrams(tokens, 2))

validation_N = sum(validation_unigram_counts.values())


def validation_bigram_prob_addk(prev, w, k):
    return (
        validation_bigram_counts[(prev, w)] + k
    ) / (
        validation_unigram_counts[(prev,)] + k * validation_V
    )


def validation_addk_probabilities(sentence, k):
    tokens = tokenize(sentence)
    return [
        validation_bigram_prob_addk(prev, word, k)
        for prev, word in get_ngrams(tokens, 2)
    ]


validation_results = []

print("Validation perplexities:")
print("-" * 40)

for k in (1, 0.5, 0.1, 0.01):
    pp = perplexity(
        [validation_sentence],
        lambda sentence, k=k: validation_addk_probabilities(sentence, k)
    )

    validation_results.append((k, pp))
    print(f"k = {k:<5} Perplexity = {pp:.4f}")


best_k_validation, validation_pp = min(
    validation_results,
    key=lambda item: item[1]
)

print(
    f"\nBest k selected using validation data: "
    f"k = {best_k_validation}"
)
print(f"Validation perplexity: {validation_pp:.4f}")


# Evaluate the validation-selected k on the original test set.
test_pp_with_validation_k = perplexity(
    test,
    make_addk_probability_function(best_k_validation)
)

print(
    f"\nTest perplexity using validation-selected k "
    f"({best_k_validation}): {test_pp_with_validation_k:.4f}"
)

print(
    f"Test perplexity using k selected directly on test "
    f"({best_k_direct}): {best_pp_direct:.4f}"
)

if best_k_validation == best_k_direct:
    print("\nThe validation-selected k is the same as the test-selected k.")
else:
    print("\nThe validation-selected k differs from the k selected directly on the test set.")


Validation perplexities:
----------------------------------------
k = 1     Perplexity = 8.5857
k = 0.5   Perplexity = 8.0163
k = 0.1   Perplexity = 8.0260
k = 0.01  Perplexity = 13.1304

Best k selected using validation data: k = 0.5
Validation perplexity: 8.0163

Test perplexity using validation-selected k (0.5): 4.5703
Test perplexity using k selected directly on test (0.01): 2.6272

The validation-selected k differs from the k selected directly on the test set.


# Final Summary

This notebook completed all nine tasks:

1. Tokenization and vocabulary construction
2. N-gram generation and formula verification
3. Unigram, bigram, and trigram counting
4. Unigram MLE model
5. Bigram MLE model and next-word prediction
6. Trigram model
7. Zero-probability problem and log probabilities
8. Add-one and add-k smoothing
9. Perplexity, model comparison, and validation-based selection of `k`

**Submitted by: Musa Sajjad — SP24-BDS-039**
